# Direct In-Notebook GGUF Text-to-Image Studio
## Generate images from text prompts

**Run Cell 1-4 from Setup notebook first!**

---
## Cell 5: Text-to-Image Generator UI

In [ ]:
# Cell 5: Direct Text-to-Image Generation Studio
import os
import sys
import torch
from pathlib import Path
from PIL import Image
import numpy as np

%store -r MASTER_DIR
%store -r MODELS_DIR
%store -r IS_COLAB

# Add ComfyUI to path
COMFYUI_DIR = f'{MASTER_DIR}/ComfyUI'
sys.path.insert(0, COMFYUI_DIR)

OUTPUT_DIR = f'{MASTER_DIR}/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("[1/3] Loading diffusion pipeline...")
try:
    from diffusers import DiffusionPipeline, AutoencoderKL
    print("✓ Diffusers loaded")
except ImportError:
    print("Installing diffusers...")
    import subprocess
    subprocess.run(['pip', 'install', 'diffusers', 'transformers'], capture_output=True)
    from diffusers import DiffusionPipeline, AutoencoderKL
    print("✓ Diffusers installed and loaded")

print("[2/3] Setting up inference device...")

device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

if device == "cuda":
    print(f"✓ Using GPU: {torch.cuda.get_device_name(0)}")
    print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}GB")
else:
    print("ℹ Using CPU (slower - recommend ZImage Turbo model)")

print("[3/3] Preparing model cache...")

# Available models for quick loading
AVAILABLE_MODELS = {
    'zimage-turbo': {
        'path': f'{MODELS_DIR}/zimage-turbo.safetensors',
        'type': 'local_safetensors',
        'speed': '⚡ Ultra-Fast',
        'quality': 'Good',
        'vram': '2-4GB'
    },
    'sdxl-turbo': {
        'path': f'{MODELS_DIR}/sdxl-turbo-q4_0.gguf',
        'type': 'gguf_quantized',
        'speed': '⚡ Fast',
        'quality': 'Excellent',
        'vram': '4-6GB'
    },
    'sd15-realistic': {
        'path': f'{MODELS_DIR}/sd15-realistic.safetensors',
        'type': 'local_safetensors',
        'speed': '🔄 Standard',
        'quality': 'Very Good',
        'vram': '4-8GB'
    },
    'pony-diffusion': {
        'path': f'{MODELS_DIR}/pony-diffusion.safetensors',
        'type': 'local_safetensors',
        'speed': '🔄 Standard',
        'quality': 'Excellent (Characters)',
        'vram': '4-8GB'
    },
}

print("\n" + "="*70)
print("AVAILABLE TEXT-TO-IMAGE MODELS")
print("="*70)

for idx, (name, info) in enumerate(AVAILABLE_MODELS.items(), 1):
    exists = "✓" if os.path.exists(info['path']) else "✗"
    print(f"{idx}. [{exists}] {name.upper():20} | {info['speed']:15} | Quality: {info['quality']}")
    print(f"   VRAM: {info['vram']}")

print("\n" + "="*70)
print("USAGE: Select model below and run generation cells")
print("="*70)

# Default model selection
SELECTED_MODEL = 'zimage-turbo'  # Change this to your preferred model

print(f"\n✓ Text-to-Image Studio Ready!")
print(f"✓ Selected Model: {SELECTED_MODEL}")
print(f"✓ Output Directory: {OUTPUT_DIR}")

%store SELECTED_MODEL
%store OUTPUT_DIR
%store AVAILABLE_MODELS
%store device
%store torch_dtype

---
## Cell 5b: Text-to-Image Generation (Run Multiple Times)

In [ ]:
# Cell 5b: Actual Text-to-Image Generation
import torch
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
from PIL import Image
import os

%store -r SELECTED_MODEL
%store -r MODELS_DIR
%store -r OUTPUT_DIR
%store -r device
%store -r torch_dtype
%store -r AVAILABLE_MODELS

# ============================================
# EDIT THESE PARAMETERS
# ============================================
prompt = "a beautiful woman with long hair, professional photography, 8k, cinematic lighting"
negative_prompt = "blurry, low quality, deformed"
num_inference_steps = 20  # Increase for higher quality (but slower)
guidance_scale = 7.5  # 0-20, higher = more adherence to prompt
height = 512
width = 512
seed = 42  # Set to None for random
num_images = 1  # Generate multiple images

print(f"\n{'='*70}")
print(f"TEXT-TO-IMAGE GENERATION")
print(f"{'='*70}")
print(f"Model: {SELECTED_MODEL}")
print(f"Device: {device}")
print(f"Prompt: {prompt}")
print(f"Steps: {num_inference_steps} | Guidance: {guidance_scale}")
print(f"Resolution: {width}x{height}")
print(f"Generating {num_images} image(s)...\n")

try:
    # Load model
    model_path = AVAILABLE_MODELS[SELECTED_MODEL]['path']
    
    if AVAILABLE_MODELS[SELECTED_MODEL]['type'] == 'local_safetensors':
        print("[1/3] Loading SafeTensors model...")
        pipe = StableDiffusionPipeline.from_single_file(
            model_path,
            torch_dtype=torch_dtype,
            safety_checker=None  # Disable safety checker for full creativity
        )
    elif AVAILABLE_MODELS[SELECTED_MODEL]['type'] == 'gguf_quantized':
        print("[1/3] Loading GGUF quantized model...")
        # GGUF models require special handling
        from diffusers import StableDiffusionPipeline
        pipe = StableDiffusionPipeline.from_pretrained(
            model_path,
            torch_dtype=torch_dtype
        )
    
    pipe = pipe.to(device)
    pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
    
    print("[2/3] Preparing generation...")
    
    # Set seed for reproducibility
    if seed is not None:
        generator = torch.Generator(device=device).manual_seed(seed)
    else:
        generator = None
    
    print("[3/3] Running inference...")
    
    with torch.no_grad():
        images = pipe(
            prompt=[prompt] * num_images,
            negative_prompt=[negative_prompt] * num_images,
            height=height,
            width=width,
            num_inference_steps=num_inference_steps,
            guidance_scale=guidance_scale,
            generator=generator
        ).images
    
    # Save images
    print(f"\n✓ Generation complete! Saving {len(images)} image(s)...")
    
    from datetime import datetime
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    for idx, img in enumerate(images, 1):
        output_path = f"{OUTPUT_DIR}/txt2img_{timestamp}_{idx}.png"
        img.save(output_path)
        print(f"  Saved: {output_path}")
        
        # Display in notebook
        if IS_COLAB:
            display(img)
        else:
            print(f"  Display: {output_path}")
    
    # Cleanup
    del pipe
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
    
except Exception as e:
    print(f"❌ Error during generation: {str(e)}")
    import traceback
    traceback.print_exc()

---
## Model Selection Reference

**For Different Use Cases:**

- **zimage-turbo**: Ultra-fast, good for testing/iteration (2.5GB)
- **sdxl-turbo**: Best quality/speed balance, quantized GGUF (2.3GB)
- **sd15-realistic**: Photorealistic people, objects (4GB)
- **pony-diffusion**: Character art, anime, highly customizable (4.2GB)

**Change `SELECTED_MODEL` above to try different models.**